# Аналитический отчет MovieLens

## Обзор
В этом отчете представлен всесторонний анализ набора данных MovieLens, включая данные о фильмах, рейтингах, тегах и внешних ссылках.
Мы рассмотрим:

фильмы, которые мы обожаем и которые критикуем,
жанры, к которым мы возвращаемся снова и снова,
теги, которые мы оставляем,
рейтинги, которые мы даем за считанные секунды,
и скрытые истины из внешних источников (IMDb — мы не забыли о вас).

Давайте рассмотрим их вместе.

## Подготовка данных

In [14]:
from movielens_analysis import Movies, Ratings, Links, Tags

def load_data(mov_path, rat_path, lin_path, tag_path):
    movies = Movies(mov_path)
    ratings = Ratings(rat_path)
    links = Links(lin_path, movies_data=movies.data)
    tags = Tags(tag_path)
    return movies, ratings, links, tags
%timeit load_data("movies.csv", "ratings.csv", "links.csv", "tags.csv")
movies, ratings, links, tags = load_data("movies.csv", "ratings.csv", "links.csv", "tags.csv")
print("Data loading completed successfully!")

7.54 ms ± 362 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
Data loading completed successfully!


## Movies Analysis

### Основные статистические данные

In [2]:
%timeit len(movies.data)
print(f"Total movies in database: {len(movies.data)}")

30 ns ± 0.806 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)
Total movies in database: 1000


### Год выхода в прокат

Когда было выпущено больше всего фильмов?
Давайте посмотрим, в какие годы было выпущено больше всего фильмов.
Ответ может оказаться неожиданным.

In [3]:
%timeit release_years = movies.dist_by_release()
release_years = movies.dist_by_release()
print("Top 10 years by movie count:")
for year, count in list(release_years.items())[:10]:
    print(f"{year}: {count} movies")

most_productive_year = next(iter(release_years))
print(f"\nMost productive year: {most_productive_year} ({release_years[most_productive_year]} movies)")

avg_per_year = sum(release_years.values()) / len(release_years)
print(f"Average movies per year: {avg_per_year:.1f}")

159 μs ± 3.11 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
Top 10 years by movie count:
1995: 224 movies
1994: 184 movies
1996: 181 movies
1993: 101 movies
1992: 23 movies
1990: 15 movies
1991: 15 movies
1989: 14 movies
1986: 9 movies
1982: 8 movies

Most productive year: 1995 (224 movies)
Average movies per year: 14.9


### Жанровый анализ

Мы часто думаем, что знаем, какие жанры популярны.
Но интуиция может нас подвести.

Пришло время узнать, какие жанры чаще всего встречаются в фильмах MovieLens?

In [4]:
%timeit genres_dist = movies.dist_by_genres()
genres_dist = movies.dist_by_genres()
print("Movie distribution by genres:")
for genre, count in genres_dist.items():
    print(f"{genre}: {count} movies")

print(f"\nMost popular genre: {next(iter(genres_dist))}")

multi_genre_count = sum(1 for m in movies.data.values() if len(m["genres"]) > 1 and '(no genres listed)' not in m["genres"])
multi_genre_percent = (multi_genre_count / len(movies.data)) * 100
print(f"Percentage of multi-genre movies: {multi_genre_percent:.1f}%")

297 μs ± 16 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
Movie distribution by genres:
Drama: 507 movies
Comedy: 365 movies
Romance: 208 movies
Thriller: 179 movies
Action: 158 movies
Adventure: 126 movies
Crime: 122 movies
Children: 100 movies
Fantasy: 69 movies
Sci-Fi: 69 movies
Mystery: 58 movies
Musical: 53 movies
Horror: 51 movies
War: 48 movies
Animation: 37 movies
Documentary: 25 movies
Western: 23 movies
Film-Noir: 18 movies
IMAX: 3 movies

Most popular genre: Drama
Percentage of multi-genre movies: 70.7%


### Фильмы с большим количеством жанров

Давайте найдем лучшие фильмы с большим количеством жанров.
Такие фильмы сложно вписать в рамки, и, возможно, именно поэтому они интересны.

In [5]:
n = 10
%timeit multi_genre_movies = movies.most_genres(n)
multi_genre_movies = movies.most_genres(n)
print(f"Top-{n} movies with most genres:")
for title, count in multi_genre_movies.items():
    print(f"{title}: {count} genres")

259 μs ± 3.78 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
Top-10 movies with most genres:
Strange Days: 6 genres
Lion King, The: 6 genres
Getaway, The: 6 genres
Super Mario Bros.: 6 genres
Beauty and the Beast: 6 genres
All Dogs Go to Heaven 2: 6 genres
Space Jam: 6 genres
Aladdin and the King of Thieves: 6 genres
Toy Story: 5 genres
Money Train: 5 genres


### Самый популярный жанр по годам (Бонус)

В какие годы какой жанр был самым популярным?
Давайте посмотрим.

In [6]:
%timeit genre_trends = movies.popular_genres(10)
genre_trends = movies.popular_genres(10)
print("Most popular genre by year:")
for year, genre in genre_trends.items():
    print(f"{year}: {genre}")

433 μs ± 16.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
Most popular genre by year:
1931: Crime
1932: Romance
1933: Comedy
1934: Comedy
1935: Comedy
1936: Comedy
1937: Drama
1938: Romance
1939: Drama
1940: Drama


## Анализ рейтингов

### Основные статистические данные

In [7]:
def data_analyze():
    len(ratings.data)
    len(set(r['movieId'] for r in ratings.data))
    len(set(r['userId'] for r in ratings.data))
%timeit data_analyze()
print(f"Total ratings loaded: {len(ratings.data)}")
print(f"Total unique movies: {len(set(r['movieId'] for r in ratings.data))}")
print(f"Total unique users: {len(set(r['userId'] for r in ratings.data))}")

76.9 μs ± 820 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
Total ratings loaded: 1000
Total unique movies: 802
Total unique users: 7


### Ежегодное распределение оценок

Каждая оценка - это небольшая реакция на фильм.
А если вы соберете их по годам? Мы увидим, в какие периоды пользователи были особенно активны.

In [8]:
ratings_movies = ratings.Movies(ratings, movies.data)
%timeit ratings_by_year = ratings_movies.dist_by_year()
ratings_by_year = ratings_movies.dist_by_year()

print("Rating distribution by year (first 5 entries):")
for year, count in list(ratings_by_year.items())[:5]:
    print(f"{year}: {count} ratings")

print(f"\nEarliest year with ratings: {min(ratings_by_year.keys())}")
print(f"Latest year with ratings: {max(ratings_by_year.keys())}")

395 μs ± 16.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
Rating distribution by year (first 5 entries):
1996: 358 ratings
1999: 82 ratings
2000: 296 ratings
2001: 70 ratings
2005: 121 ratings

Earliest year with ratings: 1996
Latest year with ratings: 2015


### Распределение оценок

Щедры ли люди на "пятерки"? Или они чаще ставят "тройки"?

Это распределение покажет, какие оценки преобладают в базе данных — низкие, высокие или средние.
Давайте выясним, что мы за критики!

In [9]:
%timeit ratings_distribution = ratings_movies.dist_by_rating()
ratings_distribution = ratings_movies.dist_by_rating()
print("Rating distribution:")
for rating, count in ratings_distribution.items():
    print(f"Rating {rating}: {count} ratings")

most_common_rating = max(ratings_distribution.items(), key=lambda x: x[1])
print(f"\nMost common rating: {most_common_rating[0]} ({most_common_rating[1]} ratings)")

73.9 μs ± 1.99 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
Rating distribution:
Rating 0.5: 24 ratings
Rating 1.0: 39 ratings
Rating 1.5: 11 ratings
Rating 2.0: 57 ratings
Rating 2.5: 7 ratings
Rating 3.0: 253 ratings
Rating 3.5: 17 ratings
Rating 4.0: 292 ratings
Rating 4.5: 33 ratings
Rating 5.0: 267 ratings

Most common rating: 4.0 (292 ratings)


### Лучшие фильмы по количеству рейтингов

Некоторые фильмы привлекают внимание и вызывают споры.
Они собирают сотни и тысячи оценок.

Здесь мы увидим лучшие фильмы, получившие наибольшее количество оценок.
Они не всегда лучшие, но, безусловно, наиболее обсуждаемые.

In [10]:
%timeit top_movies_by_ratings_count = ratings_movies.top_by_num_of_ratings(5)
top_movies_by_ratings_count = ratings_movies.top_by_num_of_ratings(5)
print("Top-5 movies by number of ratings:")
for title, count in top_movies_by_ratings_count.items():
    print(f"{title}: {count} ratings")

133 μs ± 7.34 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
Top-5 movies by number of ratings:
Usual Suspects, The (1995): 4 ratings
Pulp Fiction (1994): 4 ratings
Fugitive, The (1993): 4 ratings
Schindler's List (1993): 4 ratings
Batman (1989): 4 ratings


### Лучшие фильмы по среднему рейтингу

Какие фильмы зрители оценили лучше всего — в среднем или по медиане?

Выберите показатель и узнайте, кто стал по-настоящему великим в глазах пользователей.
И да, цифры говорят сами за себя — мы округлим их до двухзначного числа.

In [11]:
%timeit top_movies_by_avg = ratings_movies.top_by_ratings(5, 'average')
top_movies_by_avg = ratings_movies.top_by_ratings(5, 'average')
print("Top-5 movies by average rating:")
for title, avg_rating in top_movies_by_avg.items():
    print(f"{title}: {avg_rating} average rating")

302 μs ± 22.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
Top-5 movies by average rating:
Bottle Rocket (1996): 5.0 average rating
Canadian Bacon (1995): 5.0 average rating
Star Wars: Episode IV - A New Hope (1977): 5.0 average rating
James and the Giant Peach (1996): 5.0 average rating
Wizard of Oz, The (1939): 5.0 average rating


### Самые противоречивые фильмы

Иногда один и тот же фильм получает от разных людей как 1, так и 5 баллов.

Давайте посмотрим, какие фильмы вызывают наибольшую разницу во мнениях зрителей.
Именно они чаще всего становятся причиной жарких споров и дискуссий.

In [12]:
%timeit controversial_movies = ratings_movies.top_controversial(5)
controversial_movies = ratings_movies.top_controversial(5)
print("Top-5 most controversial movies (by rating variance):")
for title, variance in controversial_movies.items():
    print(f"{title}: variance {variance}")

207 μs ± 2.29 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
Top-5 most controversial movies (by rating variance):
My Fair Lady (1964): variance 5.06
Schindler's List (1993): variance 3.42
Courage Under Fire (1996): variance 3.06
Usual Suspects, The (1995): variance 2.42
Dazed and Confused (1993): variance 2.25


### Тренд рейтнгов (Бонус)

Комплексный анализ частоты и распределения тегов, содержащих определенные слова.
Возвращает словарь с детальным анализом использования ключевых слов в тегах.

In [13]:
%timeit rating_trend = ratings_movies.rating_trend_analysis(5)
rating_trend = ratings_movies.rating_trend_analysis(5)
print(rating_trend)

385 μs ± 6.11 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
{'trend': 'повышающийся', 'trend_change': 0.44, 'top_years_by_rating': {2000: 4.23, 2015: 3.95, 1999: 3.7, 1996: 3.51, 2005: 3.41}, 'yearly_averages': {1996: 3.51, 1999: 3.7, 2000: 4.23, 2001: 3.23, 2005: 3.41, 2006: 1.5, 2007: 3.0, 2011: 2.44, 2015: 3.95}, 'total_years_analyzed': 9}


### Анализ пользователей

Иногда один и тот же пользователь может ставить фильмам как восторженные 5, так и разочарованные 1 балл.

Давайте посмотрим, какие зрители демонстрируют наибольший разброс в своих оценках.
Именно их мнение часто бывает самым непредсказуемым и вызывает самые жаркие споры.

In [14]:
users_analyzer = ratings.Users(ratings)

%timeit active_users = users_analyzer.dist_by_num_of_ratings()
active_users = users_analyzer.dist_by_num_of_ratings()
print("Top-5 most active users:")
for user_id, count in list(active_users.items())[:5]:
    print(f"User {user_id}: {count} ratings")

users_by_avg = users_analyzer.dist_by_ratings('average')
print("\nTop-5 users with highest average ratings:")
for user_id, avg_rating in list(users_by_avg.items())[:5]:
    print(f"User {user_id}: {avg_rating} average rating")

users_by_variance = users_analyzer.top_controversial(5)
print("\nTop-5 users with highest rating variance:")
for user_id, variance in users_by_variance.items():
    print(f"User {user_id}: variance {variance}")

52.3 μs ± 522 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
Top-5 most active users:
User 6: 314 ratings
User 1: 232 ratings
User 4: 216 ratings
User 7: 126 ratings
User 5: 44 ratings

Top-5 users with highest average ratings:
User 1: 4.37 average rating
User 2: 3.95 average rating
User 5: 3.64 average rating
User 4: 3.56 average rating
User 6: 3.49 average rating

Top-5 users with highest rating variance:
User 3: variance 4.26
User 4: variance 1.72
User 7: variance 1.65
User 5: variance 0.96
User 6: variance 0.72


## Аналитика тегов

### Теги с большинством слов

Иногда одного слова недостаточно.
Некоторые пользователи оставляют целые фразы в качестве тегов, чтобы выразить эмоцию или мысль о фильме.

Давайте посмотрим на самые подробные теги

In [15]:
n = 5
%timeit top_tags = tags.most_words(n)
top_tags = tags.most_words(n)
print(f"Top-{n} tags with most words:")
for tag, word_count in top_tags.items():
    print(f"{tag}: {word_count} words")

156 μs ± 5.12 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
Top-5 tags with most words:
Something for everyone in this one... saw it without and plan on seeing it with kids!: 16 words
the catholic church is the most corrupt organization in history: 10 words
Oscar (Best Music - Original Score): 6 words
Everything you want is here: 5 words
based on a true story: 5 words


### Самые популярные теги

Некоторые теги появляются только один раз.
А другие встречаются снова и снова, у разных пользователей и в разных фильмах.

Давайте посмотрим, какие теги упоминались чаще всего.

In [16]:
n = 5
%timeit top_popular_tags = tags.most_popular(n)
top_popular_tags = tags.most_popular(n)
print(f"Top-{n} most popular tags:")
for tag, count in top_popular_tags.items():
    print(f"{tag}: {count} occurrences")

119 μs ± 1.31 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
Top-5 most popular tags:
funny: 15 occurrences
sci-fi: 14 occurrences
twist ending: 12 occurrences
dark comedy: 12 occurrences
atmospheric: 10 occurrences


### Самые длинные теги

Давайте посмотрим на верхние строчки самых длинных тегов по количеству символов.

In [17]:
%timeit longest_tags = tags.longest(n)
longest_tags = tags.longest(n)
print(f"Top-{n} longest tags:")
for i, tag in enumerate(longest_tags, 1):
    print(f"{i}. {tag} ({len(tag)} characters)")

116 μs ± 2.79 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
Top-5 longest tags:
1. Something for everyone in this one... saw it without and plan on seeing it with kids! (85 characters)
2. the catholic church is the most corrupt organization in history (63 characters)
3. audience intelligence underestimated (36 characters)
4. Oscar (Best Music - Original Score) (35 characters)
5. assassin-in-training (scene) (28 characters)


### Топ самых длинных по количеству слов и букв тегов

In [16]:
%timeit most_words_and_longest = tags.most_words_and_longest(10)
most_words_and_longest = tags.most_words_and_longest(10)
for _tag in most_words_and_longest:
    print(_tag)
    print()

351 μs ± 6.34 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
Something for everyone in this one... saw it without and plan on seeing it with kids!

the catholic church is the most corrupt organization in history

Oscar (Best Music - Original Score)

Everything you want is here

Guardians of the Galaxy



### Фильмы с лучшими тегами (Бонус)

У каких фильмов больше тегов? 
Давайте посмотрим 10 лучших из них.

In [18]:
%timeit top_tagged_movies = tags.tegs_movie_popular(10)
top_tagged_movies = tags.tegs_movie_popular(10)
print("Top 10 most tagged movies:")
for movie_id, tag_count in top_tagged_movies.items():
    print(f"Movie ID {movie_id}: {tag_count} tags")

56.9 μs ± 2.12 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
Top 10 most tagged movies:
Movie ID 260: 23 tags
Movie ID 135536: 19 tags
Movie ID 99114: 11 tags
Movie ID 48516: 10 tags
Movie ID 122912: 10 tags
Movie ID 7022: 10 tags
Movie ID 3176: 10 tags
Movie ID 110: 9 tags
Movie ID 7153: 9 tags
Movie ID 87430: 9 tags


### Теги, содержащие определенные слова

Иногда хочется понять, как часто пользователи упоминают что-то конкретное:
например, "забавное", "мрачное", "шедевральное" или "скучное".

Введите любое слово, и мы покажем вам все уникальные теги.

In [19]:
search_word = "Netflix"
%timeit matching_tags = tags.tags_with(search_word)
matching_tags = tags.tags_with(search_word)
print(f"Tags containing '{search_word}':")
for tag in matching_tags:
    print(f"- {tag}")

43.5 μs ± 1.27 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
Tags containing 'Netflix':
- In Netflix queue


## IMDb Анализ ссылок

Извлечение данных из IMDb
Вы хотите знать не только название фильма, но и его режиссера, бюджет, кассовые сборы и продолжительность?

Эта функция позволяет получить указанные поля из IMDb по списку MovieID.

In [17]:
sample_movie_ids = [i for i in range(1, 11)]
%timeit imdb_data = links.get_imdb(sample_movie_ids)
imdb_data = links.get_imdb(sample_movie_ids)
print(imdb_data)

20.5 s ± 5.56 s per loop (mean ± std. dev. of 7 runs, 1 loop each)
[[10, 'Martin Campbell', '$60,000,000 (estimated)', '$106,429,941', '2h 10m'], [9, 'Peter Hyams', '$35,000,000 (estimated)', '$20,350,171', '1h 51m'], [8, 'Peter Hewitt', 'Not available', '$23,920,048', '1h 37m'], [7, 'Sydney Pollack', '$58,000,000 (estimated)', '$53,672,080', '2h 7m'], [6, 'Michael Mann', '$60,000,000 (estimated)', '$67,436,818', '2h 50m'], [5, 'Charles Shyer', '$30,000,000 (estimated)', '$76,594,107', '1h 46m'], [4, 'Forest Whitaker', '$16,000,000 (estimated)', '$67,052,156', '2h 4m'], [3, 'Howard Deutch', '$25,000,000 (estimated)', '$71,518,503', '1h 41m'], [2, 'Joe Johnston', '$65,000,000 (estimated)', '$100,499,940', '1h 44m'], [1, 'John Lasseter', '$30,000,000 (estimated)', '$229,947,062', '1h 21m']]


### Лучшие режиссеры

Давайте взглянем на список самых продуктивных режиссеров по количеству снятых ими фильмов.

In [21]:
%timeit top_directors = links.top_directors(5)
top_directors = links.top_directors(5)
print("Top-5 directors:")
for director, count in top_directors.items():
    print(f"{director}: {count} movies")

1.49 μs ± 4.92 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
Top-5 directors:
Martin Campbell: 1 movies
Peter Hyams: 1 movies
Peter Hewitt: 1 movies
Sydney Pollack: 1 movies
Michael Mann: 1 movies


### Самые дорогие фильмы

Здесь представлены лучшие фильмы с самым большим бюджетом.
Показатель отражает самые дорогие фильмы из выборки.

In [22]:
%timeit most_expensive_movies = links.most_expensive(5)
most_expensive_movies = links.most_expensive(5)
print("Top-5 most expensive movies:")
for movie, budget in most_expensive_movies.items():
    print(f"{movie}: ${budget:,.0f}")

8.67 μs ± 74.6 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
Top-5 most expensive movies:
Jumanji: $65,000,000
GoldenEye: $60,000,000
Heat: $60,000,000
Sabrina: $58,000,000
Sudden Death: $35,000,000


### Самые прибыльные фильмы

Давайте посмотрим, какие фильмы принесли наибольшую прибыль,
то есть разницу между мировыми кассовыми сборами и бюджетом.

In [23]:
%timeit most_profitable_movies = links.most_profitable(5)
most_profitable_movies = links.most_profitable(5)
print("Top-5 most profitable movies:")
for movie, profit in most_profitable_movies.items():
    print(f"{movie}: ${profit:,.0f}")

14.5 μs ± 105 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
Top-5 most profitable movies:
Toy Story: $199,947,062
Waiting to Exhale: $51,052,156
Father of the Bride Part II: $46,594,107
Grumpier Old Men: $46,518,503
GoldenEye: $46,429,941


### Самые длинные фильмы

Давайте посмотрим на лучшие фильмы по продолжительности.

In [24]:
%timeit longest_movies = links.longest(5)
longest_movies = links.longest(5)
print("Top-5 longest movies:")
for movie, runtime in longest_movies.items():
    print(f"{movie}: {runtime} minutes")

11.7 μs ± 111 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
Top-5 longest movies:
Heat: 170 minutes
GoldenEye: 130 minutes
Sabrina: 127 minutes
Waiting to Exhale: 124 minutes
Sudden Death: 111 minutes


### Анализ затрат за минуту

Некоторые фильмы стоят миллионы долларов. Давайте посмотрим, сколько стоит одна минута?

В этом показателе мы делим бюджет фильма на его продолжительность
и находим лучшие фильмы с самыми дорогими затратами за минуту.

In [25]:
%timeit cost_per_minute = links.top_cost_per_minute(5)
cost_per_minute = links.top_cost_per_minute(5)
print("Top-5 movies by cost per minute:")
for movie, cost in cost_per_minute.items():
    print(f"{movie}: ${cost:.0f} per minute")

20.7 μs ± 50.7 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
Top-5 movies by cost per minute:
Jumanji: $625000 per minute
GoldenEye: $461538 per minute
Sabrina: $456693 per minute
Toy Story: $370370 per minute
Heat: $352941 per minute


### Анализ бюджета (Бонус)

Иногда интересно узнать, сколько в среднем тратится на фильм. 
На какой фильм было потрачено меньше всего или больше всего денег?

Этот показатель отражает диапазон бюджетов.

In [26]:
%timeit links.budget_analysis(10)
print(links.budget_analysis(10))

8.36 μs ± 87.9 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
Average budget: $42,111,111
Median budget: $35,000,000
Max budget: $65,000,000
Min budget: $16,000,000
